In [ ]:
"""
Objetivo: Realizar testes de raiz unitária (ADF), estimar um modelo Vetor Autorregressivo (VAR) 
e computar a matriz de Decomposição da Variância (FEVD) para analisar o risco de spillover 
entre segmentos de crédito e a macroeconomia.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. CARREGAMENTO E TRANSFORMAÇÃO DOS DADOS
# ==========================================
def carregar_e_transformar(arquivo_csv: str) -> pd.DataFrame:
    """Carrega a base, filtra a janela estendida e adiciona a variável de crescimento do PIB."""
    df = pd.read_csv(arquivo_csv, index_col=0, parse_dates=True)
    df_model = df.loc['2000-01-01':'2026-04-01'].copy()
    
    # O PIB Real é transformado em variação percentual trimestral
    df_model['pib_crescimento'] = df_model['pib_real'].pct_change() * 100
    return df_model.dropna()

# ==========================================
# 2. TESTE DE ESTACIONARIEDADE (ADF)
# ==========================================
def aplicar_estacionariedade(df: pd.DataFrame, colunas: list, p_limit: float = 0.05) -> pd.DataFrame:
    """Testa a raiz unitária para cada série e aplica a primeira diferença se necessário."""
    df_estacionario = pd.DataFrame(index=df.index)
    
    print("[*] Verificando estacionariedade (ADF)...")
    for col in colunas:
        p_value = adfuller(df[col])[1]
        if p_value > p_limit:
            df_estacionario[col] = df[col].diff()
        else:
            df_estacionario[col] = df[col]
            
    return df_estacionario.dropna()

# ==========================================
# 3. MATRIZ DE SPILLOVER E VISUALIZAÇÃO
# ==========================================
def plotar_matriz_spillover(spillover_df: pd.DataFrame, arquivo_saida: str):
    """Gera o Heatmap de Conectividade e exporta como imagem."""
    nomes_rotulos = [
        'Desemprego', 'Cresc.\ndo PIB', 'Fed\nFunds', 'Spread\n(10A-3M)', 
        'Inadimplência\n(Hipotecas)', 'Inadimplência\n(Consumidor)', 'Inadimplência\n(Cartões)'
    ]
    spillover_df.index = nomes_rotulos
    spillover_df.columns = nomes_rotulos

    plt.figure(figsize=(11, 9))
    sns.heatmap(spillover_df, annot=True, cmap='Reds', fmt='.1f', 
                cbar_kws={'label': '% da Variância Explicada'})

    plt.ylabel('Variável Afetada (Quem sofre o impacto)', fontweight='bold')
    plt.xlabel('Origem do Choque (Quem gera o impacto)', fontweight='bold')
    plt.xticks(rotation=0, fontsize=8.5)
    plt.yticks(rotation=0, fontsize=10)
    plt.tight_layout()
    plt.savefig(arquivo_saida, dpi=300)
    print(f"\n[*] Gráfico de conectividade exportado: '{arquivo_saida}'.")

if __name__ == "__main__":
    # Variáveis do sistema macrofinanceiro
    cols_var = [
        'desemprego', 'pib_crescimento', 'fed_funds', 'spread_10y3m', 
        'inad_hipotecas', 'inad_consumidor', 'inad_cartoes'
    ]
    
    # 1. Preparação da base
    df_raw = carregar_e_transformar('base_macro_fred_limpa.csv')
    df_svar = df_raw[cols_var].dropna()
    df_estacionario = aplicar_estacionariedade(df_svar, cols_var)
    
    # 2. Seleção de Lags (BIC) e Estimação VAR
    print("[*] Otimizando ordem do modelo VAR (BIC)...")
    modelo = VAR(df_estacionario)
    p_otimo = modelo.select_order(maxlags=4).bic
    resultados = modelo.fit(p_otimo)
    
    # 3. Cômputo da Decomposição da Variância (FEVD) para 8 trimestres
    horizonte = 8
    fevd = resultados.fevd(horizonte)
    
    spillover_matrix = pd.DataFrame(
        np.zeros((len(cols_var), len(cols_var))),
        index=cols_var, columns=cols_var
    )
    for i, col in enumerate(cols_var):
        spillover_matrix.iloc[i, :] = fevd.decomp[i][-1, :] * 100
        
    print("\n--- Ranking Preditivo (Impacto no Desemprego) ---")
    impacto_desemprego = spillover_matrix.loc['desemprego', ['inad_hipotecas', 'inad_consumidor', 'inad_cartoes']]
    print(impacto_desemprego.sort_values(ascending=False).to_string(float_format="{:.2f}%".format))
    
    # 4. Gráfico Final
    plotar_matriz_spillover(spillover_matrix, '01_matriz_spillover.png')